In [23]:
import pandas as pd
from astroquery.gaia import Gaia
from astropy.table import Table
import numpy as np

In [2]:
def clean_jobids():

    # Access to the Gaia Archive as registered user
    #Gaia.login()
    # Retrieve job's metadata - this may take a few minutes, depending on the number of jobs stored
    job_ids = [job.jobid for job in Gaia.list_async_jobs()]
    print('job ids',job_ids)
    # Gaia.list_async_jobs() method retrieves a "job" object that contains the job ID of each asynchronous job stored in the user account.
    # Uncomment the following line if you want to delete all the jobs stored in the user space and stop the code there. See below how to delete a subset of the jobs.
    Gaia.remove_jobs(job_ids)

In [10]:
def get_gaia_sample(catalog="gaiadr3.gold_sample_oba_stars"):
    """
    To grab the catalog star sample
    """

    query = "SELECT * FROM {}".format(catalog)

    job     = Gaia.launch_job_async(query,verbose=True)
    #job     = Gaia.launch_job(query)
    results = job.get_results()
    print(f'Table size (rows): {len(results)}')
    #results
    return results.to_pandas()

    

In [4]:
def clean_gaia_sample(targets):
    """
    Select OBA stars according to the pub A&A 674,A39 (2023) (Gaia Data Release 3: A golden sample of astrophysical parameters)
    """

    print(targets)

    # select as in paper
    idx = targets['vtan_flag'] == 0

    sel = targets[idx]

    print(len(targets),len(sel))

    return sel

In [5]:
def get_gaia_from_db(ra_min,ra_max,gaiadr='gaiadr3',catname='gaia_source'):
    # Main query, get selected infos with criteria.
    # All data information are given in the Gaia data model: 
    # https://gea.esac.esa.int/archive/documentation/GDR3/Gaia_archive/chap_datamodel/sec_dm_main_source_catalogue/ssec_dm_gaia_source.html

    vars = 'source_id, ra, dec, pmra, pmdec, parallax, parallax_error, phot_g_mean_mag, phot_bp_mean_mag, phot_rp_mean_mag, l, b, phot_variable_flag, ref_epoch'
    vars += ', L, B'
    
    if gaiadr == 'gaiadr2':
        vars += ', a_g_val'
    if gaiadr == 'gaiadr3':
        vars += ', ag_gspphot'

    """
    query = "SELECT {} \
    FROM {}.gaia_source \
    WHERE visibility_periods_used > 5 \
    AND astrometric_excess_noise < 0.5 \
    AND parallax > 1 \
    AND parallax_over_error > 5 \
    AND phot_bp_mean_flux_over_error > 20 \
    AND phot_rp_mean_flux_over_error > 20 \
    AND phot_g_mean_flux_over_error > 50 \
    AND phot_bp_rp_excess_factor < 1.2*(1.2+0.03*power(phot_bp_mean_mag-phot_rp_mean_mag,2)) \
    AND ra >= {} \
    and ra < {}".format(vars,gaiadr,ra_min,ra_max)
    """
    query = "SELECT {} \
    FROM {}.{} \
    WHERE visibility_periods_used > 8 \
    AND parallax_over_error > 10 \
    AND phot_bp_mean_flux_over_error > 20 \
    AND phot_rp_mean_flux_over_error > 20 \
    AND phot_g_mean_flux_over_error > 50 \
    AND phot_bp_rp_excess_factor < 1.3+0.06*power(phot_bp_mean_mag-phot_rp_mean_mag,2) \
    AND phot_bp_rp_excess_factor > 1.0+0.015*power(phot_bp_mean_mag-phot_rp_mean_mag,2) \
    AND astrometric_chi2_al/(astrometric_n_good_obs_al-5) < 1.44*greatest(1,exp(-0.4*(phot_g_mean_mag-19.5))) \
    AND ra >= {} \
    and ra < {}".format(vars,gaiadr,catname,ra_min,ra_max)    
    
    print(query)
    job     = Gaia.launch_job_async(query,verbose=True)
    #job     = Gaia.launch_job(query)
    results = job.get_results()
    print(f'Table size (rows): {len(results)}')
    results
    return results.to_pandas()

In [6]:
def get_sourceids(dr='gaiadr3',catalog='astrophysical_parameters'):

    vars = 'source_id'

    query = "SELECT {} FROM {}.{}".format(vars,dr,catalog)

    job     = Gaia.launch_job_async(query,verbose=True)
    #job     = Gaia.launch_job(query)
    results = job.get_results()
    print(f'Table size (rows): {len(results)}')
    results
    return results.to_pandas()

In [7]:
def get_gaia_from_dr(ra_min,ra_max,gaiadr='gaiadr3',catname='gaia_source'):
    """ 
    To get stars from a Gaia DR
    """
    
    tt = get_gaia_from_db(ra_min, ra_max,gaiadr,catname)
    ag = 'a_g_val'
    if gaiadr == 'gaiadr3':
        ag = 'ag_gspphot'
    # absolute mag G band - see Gaia Data Release 2 Documentation
    tt['MG'] = tt['phot_g_mean_mag']+5-5*np.log10(1.e3/tt['parallax'])-tt[ag]
    return tt

In [8]:
def get_targets(drs='gaiadr3',catname='gaia_source'):
    """
    To grab stars from the Gaia cat. The sky is splitted in RA slices to avoid memory pbs
    """
    
    ramin = 0
    ramax = 360
    delta_ra = 6
    ras = np.arange(ramin,ramax,delta_ra)
    outDir = '/home/philippe/LSST/gaia_files/{}/{}'.format(drs,catname)
    from sn_tools.sn_io import checkDir
    checkDir(outDir)
    for ra in ras:
        ra_min = np.round(ra,1)
        ra_max = np.round(ra_min+delta_ra,1)
        print(ra_min,ra_max)
        df = get_gaia_from_dr(ra_min,ra_max,drs,catname)
        out_name = '{}/sources_{}_{}.hdf5'.format(outDir,ra_min,ra_max)
        df.to_hdf(out_name,key='star')
        
        

In [9]:
def save_df(rr,drs,catname,key='oba_stars'):
    """
    To save a catalog
    """
    outDir = '/home/philippe/LSST/gaia_files/{}/{}'.format(drs,catname)
    from sn_tools.sn_io import checkDir
    checkDir(outDir)
    outName = 'sources_{}.hdf5'.format(catname)
    rr.to_hdf('{}/{}'.format(outDir,outName),key=key)
    

In [12]:
def get_oba_sources(catalog="gaiadr3.gold_sample_oba_stars"):
    tt_spl = table.split('.')
    drs = tt_spl[0]
    catname = tt_spl[1]
    rr = get_gaia_sample(catalog=catalog)
    save_df(rr,drs,catname,'oba_stars')
    

In [13]:
def get_fgkm_sources(catalog="gaiadr3.gold_sample_fgkm_stars"):
    tt_spl = table.split('.')
    drs = tt_spl[0]
    catname = tt_spl[1]
    rr = get_gaia_sample(catalog=catalog)
    tt = rr['evolstage_flame_spec'].unique()
    # some treatment to avoid Int32 pb...
    rr_cp = pd.DataFrame(rr)
    rr_cp["evolstage_flame_spec"] = rr_cp["evolstage_flame_spec"].apply(lambda x:-1 if x is pd.NA else x)
    #print(rr_cp['evolstage_flame_spec'].unique())
    rr_cp = rr_cp.fillna(int(-999))
    rr_cp["spectraltype_esphs"] = rr_cp["spectraltype_esphs"].apply(lambda x:'U' if x=='' else x)
    print(rr_cp['spectraltype_esphs'].unique())
    rr_cp['spectraltype_esphs'] = rr_cp['spectraltype_esphs'].astype(str)
    rr_cp['evolstage_flame'] = rr_cp['evolstage_flame'].astype(int)
    print(rr_cp.dtypes)
    save_df(rr_cp,drs,catname,'fgkm_stars')

In [31]:
def get_meta(df,Gaia,cols='ap.*'):
    
    table_id = Table([list(df['source_id'])], names=['gaia_id'], meta={'meta':'table'})
    Gaia.upload_table(upload_resource=table_id, table_name='tableid')

    query="SELECT {} \
    FROM gaiadr3.astrophysical_parameters AS ap \
    JOIN user_pgris.tableid as usert ON usert.gaia_id = ap.source_id".format(cols)

    #launch query and save and return a Dataframe
    job = Gaia.launch_job_async(query)
    results = job.get_results().to_pandas()

    # suggest delete your table from your personal space
    Gaia.delete_user_table(table_name='tableid')
    return results

In [72]:
def get_sample_from_astro_params(gaiaDir = '../../gaia_files',gaiadr = 'gaiadr3',
                                 astro_cat = 'gold_sample_oba_stars/sources_gold_sample_oba_stars.hdf5',clean_stars=True):
    # get source_id's

    stars = pd.read_hdf('{}/{}/{}'.format(gaiaDir,gaiadr,astro_cat))

    if clean_stars:
        stars = clean_gaia_sample(stars)
    len(stars)
    stars['source_id'].to_list()
    # make a request to grab available columns
    rr = get_meta(stars[:10],Gaia)
    cols = rr.columns.to_list()
    cols = list(filter(lambda s: not ('oa' in s), cols))
    cols_db = ','.join(cols)
    cols_db
    arr_spl = np.array_split(stars, 20)
    df_fi = pd.DataFrame()
    clean_jobids()
    Gaia.delete_user_table(table_name='tableid')
    for i,vv in enumerate(arr_spl):
        print(i,len(vv))
        dfb = get_meta(vv,Gaia,cols_db)
        df_fi = pd.concat((df_fi,dfb))
    #pb with Int32 dtype when saving to hdf file
    lInt = list(df_fi.select_dtypes(['Int32']).columns)
    for vv in lInt:
        df_fi[vv] = df_fi[vv].astype(int)
    save_df(df_fi,'gaiadr3','gold_sample_oba_astrophysical_parameters')    
    

In [81]:
def get_sample_from_gaia_source(gaiaDir = '../../gaia_files',gaiadr = 'gaiadr3',
                                 astro_cat = 'gold_sample_oba_stars/sources_gold_sample_oba_stars.hdf5',
                                 source_dir='gaia_source',
                                 clean_stars=True):
    # get source_id's

    stars = pd.read_hdf('{}/{}/{}'.format(gaiaDir,gaiadr,astro_cat))

    if clean_stars:
        stars = clean_gaia_sample(stars)
    len(stars)
    stars_source_id = stars['source_id'].to_list()
    import glob
    fName='{}/{}/{}/*.hdf5'.format(gaiaDir,gaiadr,source_dir)
    fis = glob.glob(fName)
    df_fi = pd.DataFrame()
    for fi in fis:
        print('processing',fi)
        da = pd.read_hdf(fi)
        idx = da['source_id'].isin(stars_source_id)
        sel = da[idx]
        if len(sel) > 0:
            df_fi=pd.concat((df_fi,sel))

    print(df_fi)
    save_df(df_fi,'gaiadr3','gold_sample_oba_gaia_source')
    

In [34]:
Gaia.login(user='pgris', password='Lsst!!2024a=+')

INFO: Login to gaia TAP server [astroquery.gaia.core]
INFO: OK [astroquery.utils.tap.core]
INFO: Login to gaia data server [astroquery.gaia.core]
INFO: OK [astroquery.utils.tap.core]


In [34]:
#grab OBA catalog and save it on disk
# get_oba_sources()

Launched query: 'SELECT * FROM gaiadr3.gold_sample_oba_stars'
------>https
host = gea.esac.esa.int:443
context = /tap-server/tap/async
Content-type = application/x-www-form-urlencoded
303 303
[('Date', 'Tue, 21 Oct 2025 06:42:26 GMT'), ('Server', 'Apache/2.4.6 (SLES Expanded Support platform 7) OpenSSL/1.0.2k-fips mod_jk/1.2.43'), ('X-VO-Authenticated', 'pgris'), ('Location', 'https://gea.esac.esa.int/tap-server/tap/async/1761028946091O'), ('Cache-Control', 'no-cache, no-store, max-age=0, must-revalidate'), ('Pragma', 'no-cache'), ('Expires', '0'), ('X-XSS-Protection', '1; mode=block'), ('X-Frame-Options', 'SAMEORIGIN'), ('X-Content-Type-Options', 'nosniff'), ('Transfer-Encoding', 'chunked'), ('Content-Type', 'text/plain;charset=ISO-8859-1')]
job 1761028946091O, at: https://gea.esac.esa.int/tap-server/tap/async/1761028946091O
Retrieving async. results...
INFO: Query finished. [astroquery.utils.tap.core]
Table size (rows): 3023388


In [14]:
#grab FGKM stars and dump on disk
# get_fgkm_sources()

In [78]:
# get OBA stars in astrophysical_parameters catalog
#get_sample_from_astro_params()

In [82]:
get_sample_from_gaia_source()

                   source_id  vtan_flag
0         243212732278284416          0
1         243215068740428672          0
2         243222662242499840          0
3         243225651539759104          0
4         243225685899488640          0
...                      ...        ...
3023383  5959547056346917248          0
3023384  5959547567393419392          0
3023385  5959547670472310528          0
3023386  5959549255370331392          0
3023387  5959549392809229824          0

[3023388 rows x 2 columns]
3023388 2746935
processing ../../gaia_files/gaiadr3/gaia_source/sources_312_318.hdf5
processing ../../gaia_files/gaiadr3/gaia_source/sources_234_240.hdf5
processing ../../gaia_files/gaiadr3/gaia_source/sources_30_36.hdf5
processing ../../gaia_files/gaiadr3/gaia_source/sources_126_132.hdf5
processing ../../gaia_files/gaiadr3/gaia_source/sources_162_168.hdf5
processing ../../gaia_files/gaiadr3/gaia_source/sources_144_150.hdf5
processing ../../gaia_files/gaiadr3/gaia_source/sources_216_222.

In [15]:
targets.to_hdf('targets_OBA.hdf5',key='gaiadr3')

In [16]:
sel.to_hdf('OBA_gloden.hdf5',key='gaiadr3')